# NLA Reconstructor — Batch
Lee `explicaciones_nla.csv` (salida de `NLA_verbalizer_batch.ipynb`) y los `.npy` de activaciones,
reconstruye cada activación desde su explicación y calcula cosine similarity + MSE.
Produce `resultados_nla.csv` con todas las métricas listas para el análisis de sesgos.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
import torch, numpy as np, json, os, yaml, csv
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from safetensors.torch import load_file


drive.mount('/content/drive')

# ── Rutas ── deben coincidir con los notebooks anteriores
HACKATHON      = '/content/drive/MyDrive/HACKATHON'
CHECKPOINT_AR  = '/content/drive/MyDrive/nla_pipeline/checkpoints/nla_ar'
DIR_ACT        = f'{HACKATHON}/activaciones'
CSV_VERB       = f'{HACKATHON}/explicaciones_nla.csv'   # salida del verbalizer batch
CSV_SALIDA     = f'{HACKATHON}/resultados_nla.csv'

# Cargar configuración del AR
with open(f'{CHECKPOINT_AR}/nla_meta.yaml') as f:
    ar_meta = yaml.safe_load(f)

MSE_SCALE  = ar_meta['extraction']['mse_scale']   # √3584 ≈ 59.87
AR_TEMPLATE = ar_meta['prompt_templates']['ar']
print(f'mse_scale   : {MSE_SCALE:.4f}')
print(f'AR template : {AR_TEMPLATE}')

# Cargar CSV del verbalizer
with open(CSV_VERB, encoding='utf-8') as f:
    reader  = csv.DictReader(f)
    filas_verb = list(reader)

print(f'\nEntradas cargadas desde verbalizer : {len(filas_verb)}')
print(f'Carpeta .npy                       : {DIR_ACT}')
print(f'CSV de salida                      : {CSV_SALIDA}')

# Detectar VRAM
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
MODO_8BIT = vram_gb < 20
print(f'\nVRAM: {vram_gb:.1f} GB → Modo: {"8-bit" if MODO_8BIT else "bfloat16"}')

In [ ]:
# Cargar AR
tok_ar = AutoTokenizer.from_pretrained(CHECKPOINT_AR, trust_remote_code=True)

print('Cargando AR...')
if MODO_8BIT:
    ar_backbone = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_AR,
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map='auto', trust_remote_code=True,
    )
else:
    ar_backbone = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_AR, torch_dtype=torch.bfloat16,
        device_map='cuda:0', trust_remote_code=True,
    )

# Quitar LayerNorm final (el AR opera sobre la salida raw del bloque)
inner = ar_backbone.model
for attr in ('norm', 'final_layernorm', 'ln_f'):
    if hasattr(inner, attr):
        setattr(inner, attr, torch.nn.Identity())
        print(f'✓ LayerNorm final ({attr}) → Identity')
        break

# Cargar value_head
d = ar_backbone.config.hidden_size   # 3584
value_head = torch.nn.Linear(d, d, bias=False, dtype=torch.float32)
value_head.load_state_dict(load_file(f'{CHECKPOINT_AR}/value_head.safetensors'))
device = next(ar_backbone.parameters()).device
value_head = value_head.to(device).eval()

ar_backbone.eval()
print(f'✓ AR listo. VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
@torch.inference_mode()
def reconstruir(descripcion):
    """Texto → vector reconstruido [3584] vía AR."""
    prompt = AR_TEMPLATE.format(explanation=descripcion)
    ids    = tok_ar(prompt, return_tensors='pt',
                    add_special_tokens=True)['input_ids'].to(device)
    h      = ar_backbone.model(ids, use_cache=False).last_hidden_state
    pred   = value_head(h[0, -1].float()).cpu()   # último token → [3584]
    return pred


def score(descripcion, v_raw_np):
    """Retorna (mse, cos_sim) entre vector reconstruido y original."""
    pred   = reconstruir(descripcion)
    gold   = torch.as_tensor(v_raw_np, dtype=torch.float32)

    pred_n = pred / pred.norm().clamp_min(1e-12) * MSE_SCALE
    gold_n = gold / gold.norm().clamp_min(1e-12) * MSE_SCALE

    mse = ((pred_n - gold_n) ** 2).mean().item()
    cos = torch.nn.functional.cosine_similarity(
        pred.unsqueeze(0), gold.unsqueeze(0)
    ).item()
    return mse, cos


def interpretar(cos):
    if cos >= 0.9:    return 'EXCELENTE'
    elif cos >= 0.75: return 'BUENO'
    elif cos >= 0.5:  return 'MEDIOCRE'
    else:             return 'POBRE'


print('✓ Funciones reconstruir(), score() e interpretar() listas')

In [ ]:
# ── Loop batch ──────────────────────────────────────────────────────
COLUMNAS = [
    'id', 'lang', 'grupo', 'tema', 'texto',
    'senales_colombianas', 'hipotesis_nla',
    'explicacion_nla', 'cos_sim', 'mse', 'fidelidad',
]

resultados = []
errores    = []
total      = len(filas_verb)

print(f'Evaluando {total} entradas...\n')

for i, fila in enumerate(filas_verb):
    pid  = fila['id']
    lang = fila['lang']
    print(f'[{i+1:3d}/{total}] {pid}_{lang} ...', end=' ', flush=True)

    try:
        # Cargar activación original
        v_raw = np.load(f'{DIR_ACT}/{pid}_{lang}.npy')   # [3584]

        mse, cos = score(fila['explicacion_nla'], v_raw)

        resultados.append({
            'id'                 : pid,
            'lang'               : lang,
            'grupo'              : fila['grupo'],
            'tema'               : fila['tema'],
            'texto'              : fila['texto'],
            'senales_colombianas': fila['senales_colombianas'],
            'hipotesis_nla'      : fila['hipotesis_nla'],
            'explicacion_nla'    : fila['explicacion_nla'],
            'cos_sim'            : round(cos, 4),
            'mse'                : round(mse, 4),
            'fidelidad'          : interpretar(cos),
        })
        print(f'cos={cos:.3f} | {interpretar(cos)}')

    except Exception as e:
        errores.append({'id': pid, 'lang': lang, 'error': str(e)})
        print(f'✗ ERROR: {e}')

# ── Guardar CSV final ───────────────────────────────────────────────
with open(CSV_SALIDA, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNAS)
    writer.writeheader()
    writer.writerows(resultados)

# Resumen estadístico por grupo
from collections import defaultdict
por_grupo = defaultdict(list)
for r in resultados:
    por_grupo[r['grupo']].append(r['cos_sim'])

print(f'\n{"="*55}')
print('RESUMEN POR GRUPO')
print(f'{"="*55}')
for grupo, vals in sorted(por_grupo.items()):
    print(f'{grupo}')
    print(f'  n={len(vals)}  cos_media={sum(vals)/len(vals):.4f}  '
          f'max={max(vals):.4f}  min={min(vals):.4f}')

print(f'\n✓ Resultados guardados : {len(resultados)}')
print(f'✗ Errores              : {len(errores)}')
if errores:
    for e in errores:
        print(f'   {e}')
print(f'✓ CSV → {CSV_SALIDA}')
print(f'\n🎉 Pipeline NLA completado. Procede con el análisis de sesgos.')